In [125]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dotenv
import os
dotenv.load_dotenv()

True

In [130]:
DIR = os.getenv("PROJECT_DIR")
df = pd.read_csv(DIR+"/data/raw/nyc311_2023.csv")

/var/folders/f9/wdmym5p11jbg80pp8ll6rsfc0000gn/T/ipykernel_27030/3097476470.py:2: DtypeWarning: Columns (0: vehicle_type, 1: taxi_company_borough) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DIR+"/data/raw/nyc311_2023.csv")


In [127]:
df.info()
Starting_length = len(df)

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 44 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   unique_key                      150000 non-null  int64  
 1   created_date                    150000 non-null  str    
 2   closed_date                     149290 non-null  str    
 3   agency                          150000 non-null  str    
 4   agency_name                     150000 non-null  str    
 5   complaint_type                  150000 non-null  str    
 6   descriptor                      147056 non-null  str    
 7   location_type                   129700 non-null  str    
 8   incident_zip                    148043 non-null  float64
 9   incident_address                142800 non-null  str    
 10  street_name                     142800 non-null  str    
 11  cross_street_1                  96923 non-null   str    
 12  cross_street_2             

In [83]:
null_rates = df.isnull().sum() / len(df) * 100
print(null_rates.sort_values(ascending=False))

vehicle_type                      99.980000
taxi_company_borough              99.947333
due_date                          99.794667
road_ramp                         99.755333
bridge_highway_direction          99.665333
facility_type                     99.412000
bridge_highway_segment            99.339333
bridge_highway_name               99.338000
taxi_pick_up_location             99.030667
descriptor_2                      56.066000
landmark                          46.846000
intersection_street_1             39.567333
intersection_street_2             39.465333
cross_street_1                    35.384667
cross_street_2                    35.316000
location_type                     13.533333
bbl                               11.443333
incident_address                   4.800000
street_name                        4.800000
city                               4.350667
council_district                   2.394667
descriptor                         1.962667
longitude                       

In [84]:
print(f"Full duplicates: {df.duplicated().sum()}")
print(f"Duplicate unique_key: {df['unique_key'].duplicated().sum()}")

Full duplicates: 0
Duplicate unique_key: 0


In [85]:
print(df[['created_date', 'closed_date']].dtypes)
print(df['created_date'].head(3))

created_date    str
closed_date     str
dtype: object
0    2023-01-01T00:00:00.000
1    2023-01-01T00:00:09.000
2    2023-01-01T00:00:42.000
Name: created_date, dtype: str


In [86]:
for col in ['complaint_type', 'agency', 'city', 'borough', 'status']:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(10))


complaint_type: 167 unique values
complaint_type
Illegal Parking            23123
HEAT/HOT WATER             19204
Noise - Residential        13311
Blocked Driveway            8177
UNSANITARY CONDITION        5376
Street Condition            4104
PLUMBING                    3381
Abandoned Vehicle           3379
Noise - Street/Sidewalk     3174
Water System                2862
Name: count, dtype: int64

agency: 14 unique values
agency
NYPD     59239
HPD      40600
DSNY     12657
DOT      11674
DEP       7929
DOB       4533
DOHMH     3835
DPR       3463
EDC       2063
DHS       1480
Name: count, dtype: int64

city: 61 unique values
city
BROOKLYN         44867
BRONX            31252
NEW YORK         26999
STATEN ISLAND     6423
JAMAICA           3108
ASTORIA           2439
FLUSHING          2275
QUEENS            2260
RIDGEWOOD         1912
MANHATTAN         1301
Name: count, dtype: int64

borough: 6 unique values
borough
BROOKLYN         46427
QUEENS           34407
BRONX            320

Dropping the columns where the null rate is equal or greater than 90

In [87]:
cols_to_drop = null_rates[null_rates >= 90].index.tolist()
print(cols_to_drop)

['taxi_pick_up_location', 'facility_type', 'bridge_highway_name', 'bridge_highway_segment', 'road_ramp', 'bridge_highway_direction', 'vehicle_type', 'taxi_company_borough', 'due_date']


In [88]:
count = 1
for key, value in null_rates.items():
    if value >= 90:
        if count==1:
            print("--"*10)
            print("|Name| Null rate(%)|")
            print("--"*10)
        print(f"|{key} | {round(value,2)}|")
        count+=1

--------------------
|Name| Null rate(%)|
--------------------
|taxi_pick_up_location | 99.03|
|facility_type | 99.41|
|bridge_highway_name | 99.34|
|bridge_highway_segment | 99.34|
|road_ramp | 99.76|
|bridge_highway_direction | 99.67|
|vehicle_type | 99.98|
|taxi_company_borough | 99.95|
|due_date | 99.79|


In [89]:
df = df.drop(cols_to_drop,axis=1)

In [90]:
df

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,...,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,latitude,longitude,location,descriptor_2
0,56416396,2023-01-01T00:00:00.000,2023-01-03T14:56:48.000,DOHMH,Department of Health and Mental Hygiene,Food Poisoning,1 or 2,Restaurant/Bar/Deli/Bakery,11379.0,84-46 ELIOT AVENUE,...,QUEENS,1019280.0,204397.0,ONLINE,Unspecified,QUEENS,40.727630,-73.873614,"{'type': 'Point', 'coordinates': [-73.87361356...",NaN
1,56417527,2023-01-01T00:00:09.000,2023-01-01T00:36:06.000,NYPD,New York City Police Department,Illegal Fireworks,NaN,Street/Sidewalk,11218.0,AVENUE C,...,BROOKLYN,991565.0,172780.0,ONLINE,Unspecified,BROOKLYN,40.640915,-73.973642,"{'type': 'Point', 'coordinates': [-73.97364216...",NaN
2,56416252,2023-01-01T00:00:42.000,2023-01-01T17:34:15.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10453.0,1871 SEDGWICK AVENUE,...,BRONX,1007015.0,250368.0,ONLINE,Unspecified,BRONX,40.853848,-73.917709,"{'type': 'Point', 'coordinates': [-73.91770920...",NaN
3,56418795,2023-01-01T00:00:45.000,2023-01-01T01:24:10.000,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,10001.0,15 HUDSON BOULEVARD,...,MANHATTAN,984043.0,214298.0,MOBILE,Unspecified,MANHATTAN,40.754875,-74.000747,"{'type': 'Point', 'coordinates': [-74.00074715...",NaN
4,56418136,2023-01-01T00:00:46.000,2023-01-01T01:01:43.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,11234.0,1621 EAST 51 STREET,...,BROOKLYN,1004782.0,165410.0,ONLINE,Unspecified,BROOKLYN,40.620665,-73.926040,"{'type': 'Point', 'coordinates': [-73.92604033...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,56580689,2023-01-20T08:59:09.000,2023-06-05T12:04:22.000,TLC,Taxi and Limousine Commission,For Hire Vehicle Complaint,Driver Complaint - Non Passenger,Street,11201.0,FLATBUSH AVENUE EXTENSION,...,BROOKLYN,989092.0,191410.0,PHONE,Unspecified,BROOKLYN,40.692052,-73.982540,"{'type': 'Point', 'coordinates': [-73.98253964...",Unsafe Driving
149996,56583198,2023-01-20T08:59:12.000,2023-01-20T09:41:42.000,NYPD,New York City Police Department,Illegal Parking,Blocked Bike Lane,Street/Sidewalk,10458.0,222 EAST KINGSBRIDGE ROAD,...,BRONX,1013637.0,254594.0,ONLINE,Unspecified,BRONX,40.865427,-73.893754,"{'type': 'Point', 'coordinates': [-73.89375352...",NaN
149997,56578792,2023-01-20T08:59:34.000,2023-01-20T11:59:43.000,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,Street/Sidewalk,11101.0,37 AVENUE,...,QUEENS,1004764.0,213294.0,MOBILE,Unspecified,QUEENS,40.752095,-73.925959,"{'type': 'Point', 'coordinates': [-73.92595937...",NaN
149998,56577566,2023-01-20T08:59:47.000,2023-01-20T12:11:27.000,NYPD,New York City Police Department,Blocked Driveway,Partial Access,Street/Sidewalk,11105.0,45-11 23 AVENUE,...,QUEENS,1010521.0,219436.0,ONLINE,Unspecified,QUEENS,40.768938,-73.905157,"{'type': 'Point', 'coordinates': [-73.90515680...",NaN


Dropping "Unspecified" in bourough and status with null

In [91]:
print(f"Bourough(containing Unspecified) {len(df[df['borough']=="Unspecified"])}")
print(f"Status(containing Unspecified) {len(df[df['status']=="Unspecified"])}")



Bourough(containing Unspecified) 449
Status(containing Unspecified) 35


In [92]:
df = df[ df['borough']!= "Unspecified"]
df = df[ df['status']!= "Unspecified"]
print(f"Length of the dataframe {len(df)}")

Length of the dataframe 149516


Standardizing the data since complaint_type data contains irregular data

In [93]:
df['complaint_type'] = df['complaint_type'].str.title()

converting the created_date,closed_date to datetime

In [94]:
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

In [95]:
df['resolution_time'] = (df['closed_date'] - df['created_date'])
df.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,...,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,latitude,longitude,location,descriptor_2,resolution_time
0,56416396,2023-01-01 00:00:00,2023-01-03 14:56:48,DOHMH,Department of Health and Mental Hygiene,Food Poisoning,1 or 2,Restaurant/Bar/Deli/Bakery,11379.0,84-46 ELIOT AVENUE,...,1019280.0,204397.0,ONLINE,Unspecified,QUEENS,40.727630,-73.873614,"{'type': 'Point', 'coordinates': [-73.87361356...",NaN,2 days 14:56:48
1,56417527,2023-01-01 00:00:09,2023-01-01 00:36:06,NYPD,New York City Police Department,Illegal Fireworks,NaN,Street/Sidewalk,11218.0,AVENUE C,...,991565.0,172780.0,ONLINE,Unspecified,BROOKLYN,40.640915,-73.973642,"{'type': 'Point', 'coordinates': [-73.97364216...",NaN,0 days 00:35:57
2,56416252,2023-01-01 00:00:42,2023-01-01 17:34:15,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10453.0,1871 SEDGWICK AVENUE,...,1007015.0,250368.0,ONLINE,Unspecified,BRONX,40.853848,-73.917709,"{'type': 'Point', 'coordinates': [-73.91770920...",NaN,0 days 17:33:33
3,56418795,2023-01-01 00:00:45,2023-01-01 01:24:10,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,10001.0,15 HUDSON BOULEVARD,...,984043.0,214298.0,MOBILE,Unspecified,MANHATTAN,40.754875,-74.000747,"{'type': 'Point', 'coordinates': [-74.00074715...",NaN,0 days 01:23:25
4,56418136,2023-01-01 00:00:46,2023-01-01 01:01:43,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,11234.0,1621 EAST 51 STREET,...,1004782.0,165410.0,ONLINE,Unspecified,BROOKLYN,40.620665,-73.926040,"{'type': 'Point', 'coordinates': [-73.92604033...",NaN,0 days 01:00:57


In [96]:
print(f"The percentage of null latitudes: {round(df['latitude'].isna().sum()/len(df)*100)}%")
print(f"The percentage of null longitudes: {round(df['longitude'].isna().sum()/len(df)*100)}%")


The percentage of null latitudes: 2%
The percentage of null longitudes: 2%


Only 2% of these columns is null so dropping it entirely is the next best option

In [97]:
df = df[df['latitude'].isnull() != True]
df = df[df['longitude'].isnull() != True]
print(f"length after dropping the data{len(df)}")


length after dropping the data147082


In [98]:
df = df.drop(columns=['descriptor_2'])

In [99]:
df[df['location_type'].isnull()]['complaint_type'].value_counts().head(10)

complaint_type
Water System                     2832
Street Light Condition           2479
Noise                            2429
Traffic Signal Condition         1929
Street Condition                 1816
General Construction/Plumbing    1722
Sewer                            1261
Building/Use                      823
Elevator                          793
Air Quality                       508
Name: count, dtype: int64

In [108]:
df = df[df['resolution_time'] >= pd.Timedelta(seconds=0)]
print(f"Rows remaining: {len(df)}")

Rows remaining: 146064


In [103]:
print(f"Starting data length {Starting_length} | Ending length after cleaning the data {len(df)}")

Starting data length 150000 | Ending length after cleaning the data 146064


In [106]:
# Row counts
print(f"Final row count: {len(df)}")

# Null rates after cleaning
null_rates_clean = df.isnull().sum() / len(df) * 100
print(null_rates_clean.sort_values(ascending=False))

# Date range confirmation
print(f"Date range: {df['created_date'].min()} to {df['created_date'].max()}")

# Negative resolution times
print(f"Negative resolution times: {(df['resolution_time'] <= pd.Timedelta(0)).sum()}")

# Duplicate check
print(f"Duplicate unique_key: {df['unique_key'].duplicated().sum()}")

Final row count: 146064
landmark                          45.742277
intersection_street_1             39.147908
intersection_street_2             39.063013
cross_street_1                    35.246193
cross_street_2                    35.176361
location_type                     12.529439
bbl                                9.485568
incident_address                   3.992086
street_name                        3.992086
city                               3.241045
descriptor                         1.984062
incident_zip                       0.490196
council_district                   0.490196
address_type                       0.217713
resolution_description             0.097218
park_facility_name                 0.039709
resolution_action_updated_date     0.004108
location                           0.000000
longitude                          0.000000
borough                            0.000000
open_data_channel_type             0.000000
y_coordinate_state_plane           0.000000
latitude

In [ ]:
print("Cleaned columns",len(df.columns))

Cleaned columns 35


In [109]:
print(f"Negative resolution times remaining: {(df['resolution_time'] < pd.Timedelta(seconds=0)).sum()}")

Negative resolution times remaining: 0


In [113]:
df.to_csv("cleaned_nyc_311_dataset.csv",index=False)
print(f"Exported succesfull: {len(df)} rows")

Exported succesfull: 146064 rows
